# API Test Notebook
This notebook calls RF-DETR and Gemma4 APIs hosted on your server.

In [15]:
import requests
from pprint import pprint

RF_DETR_BASE = "http://129.97.170.141:8000"
GEMMA4_BASE = "http://129.97.170.141:8001"
IMAGE_URL = "https://media.roboflow.com/dog.jpg"
TIMEOUT = 120

In [16]:
# Health checks
rf_health = requests.get(f"{RF_DETR_BASE}/health", timeout=TIMEOUT).json()
gemma_health = requests.get(f"{GEMMA4_BASE}/health", timeout=TIMEOUT).json()

print("RF-DETR health:")
pprint(rf_health)
print()
print("Gemma4 health:")
pprint(gemma_health)

RF-DETR health:
{'cuda_visible_devices': '0', 'status': 'ok'}

Gemma4 health:
{'cuda_visible_devices': '1',
 'model_id': 'google/gemma-4-E2B-it',
 'status': 'ok'}


In [17]:
# RF-DETR POST /detect
detect_payload = {
    "image_url": IMAGE_URL,
    "threshold": 0.5,
    "save_annotated": False
}

detect_raw_response = requests.post(
    f"{RF_DETR_BASE}/detect",
    json=detect_payload,
    timeout=TIMEOUT,
)
detect_raw_response.raise_for_status()
detect_response = detect_raw_response.json()

print("RF-DETR detect response:")
pprint(detect_response)

RF-DETR detect response:
{'count': 3,
 'detections': [{'bbox_xyxy': [68.82357025146484,
                               247.8577880859375,
                               621.8697509765625,
                               926.580810546875],
                 'class_id': 18,
                 'class_name': 'dog',
                 'confidence': 0.9337944388389587},
                {'bbox_xyxy': [626.3731689453125,
                               731.4295654296875,
                               696.5244140625,
                               787.971923828125],
                 'class_id': 3,
                 'class_name': 'car',
                 'confidence': 0.6154981255531311},
                {'bbox_xyxy': [0.8009076118469238,
                               354.81591796875,
                               647.8016357421875,
                               1265.427490234375],
                 'class_id': 1,
                 'class_name': 'person',
                 'confidence': 0.53368419408798

In [18]:
# Gemma4 POST /generate with same image + RF-DETR output
generate_payload = {
    "prompt": "Describe this image using the RF-DETR detections. Keep it concise.",
    "image_url": IMAGE_URL,
    "rf_detr_output": detect_response,
    "max_new_tokens": 196
}

generate_raw_response = requests.post(
    f"{GEMMA4_BASE}/generate",
    json=generate_payload,
    timeout=TIMEOUT,
)
generate_raw_response.raise_for_status()
generate_response = generate_raw_response.json()

print("Gemma4 generate response:")
pprint(generate_response)

Gemma4 generate response:
{'response': {'content': 'Please provide the image you are referring to. I '
                         'need an image in order to describe it using RF-DETR '
                         'detections.',
              'role': 'assistant'}}
